Spark reads and writes data through a unified API — `spark.read` for loading, `df.write` for saving. The format, path, and a handful of options are all that change between CSV, JSON, Parquet, ORC, and Delta Lake. This notebook reads all eight Fintech Bank tables from `data/` and demonstrates writing patterns for each format.

## Spark Data Sources

![Spark Data Sources](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-data-sources.svg)

All formats share the same fluent API:

```python
# Read
df = spark.read.format("parquet").option("…", "…").load("path/")

# Write
df.write.format("parquet").mode("overwrite").save("path/")
```

The `.format()` shorthand methods — `.csv()`, `.json()`, `.parquet()`, `.orc()` — are aliases for the above.

## Setup

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, BooleanType,
    DecimalType, DateType, TimestampType, DoubleType,
)

spark = (
    SparkSession.builder
    .appName("ReadingWritingData")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

REPO_ROOT = os.path.dirname(os.path.abspath("07-reading-writing-data.ipynb"))
DATA      = os.path.join(REPO_ROOT, "data")
OUT       = os.path.join(REPO_ROOT, "data", "out")   # scratch output folder

print(f"DATA : {DATA}")
print(f"OUT  : {OUT}")

## Reading CSV

`option("header", "true")` reads the first row as column names. Always pass an explicit schema — inference requires a full data scan and may get types wrong (e.g. `credit_score` as `bigint` instead of `int`).

In [ ]:
customers_schema = StructType([
    StructField("customer_id",   StringType(),    nullable=False),
    StructField("full_name",     StringType(),    nullable=False),
    StructField("email",         StringType(),    nullable=True),
    StructField("phone",         StringType(),    nullable=True),
    StructField("city",          StringType(),    nullable=True),
    StructField("state",         StringType(),    nullable=True),
    StructField("date_of_birth", DateType(),      nullable=True),
    StructField("kyc_verified",  BooleanType(),   nullable=False),
    StructField("created_at",    TimestampType(), nullable=False),
])

customers = (
    spark.read
    .option("header", "true")
    .schema(customers_schema)
    .csv(f"{DATA}/customers")
)

print(f"customers: {customers.count()} rows")
customers.printSchema()
customers.show(5, truncate=False)

In [ ]:
loan_repayments_schema = StructType([
    StructField("repayment_id", StringType(),       nullable=False),
    StructField("loan_id",      StringType(),       nullable=False),
    StructField("customer_id",  StringType(),       nullable=False),
    StructField("due_date",     DateType(),         nullable=False),
    StructField("paid_date",    DateType(),         nullable=True),
    StructField("emi_amount",   DecimalType(18, 2), nullable=False),
    StructField("paid_amount",  DecimalType(18, 2), nullable=True),
    StructField("status",       StringType(),       nullable=False),
])

loan_repayments = (
    spark.read
    .option("header", "true")
    .schema(loan_repayments_schema)
    .csv(f"{DATA}/loan_repayments")
)

print(f"loan_repayments: {loan_repayments.count()} rows")
loan_repayments.show(5)

## Reading JSON

JSON files don't need a header option — keys become column names automatically. Use `option("multiLine", "true")` when each document spans multiple lines (common for loan origination system payloads).

In [ ]:
card_accounts_schema = StructType([
    StructField("card_id",      StringType(),       nullable=False),
    StructField("customer_id",  StringType(),       nullable=False),
    StructField("card_type",    StringType(),       nullable=False),
    StructField("network",      StringType(),       nullable=True),
    StructField("credit_limit", DecimalType(18, 2), nullable=True),
    StructField("issued_on",    DateType(),         nullable=False),
    StructField("expiry_date",  DateType(),         nullable=False),
    StructField("status",       StringType(),       nullable=False),
])

card_accounts = (
    spark.read
    .schema(card_accounts_schema)
    .json(f"{DATA}/card_accounts")
)

print(f"card_accounts: {card_accounts.count()} rows")
card_accounts.show(5)

In [ ]:
loan_accounts_schema = StructType([
    StructField("loan_id",       StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("loan_type",     StringType(),       nullable=False),
    StructField("principal",     DecimalType(18, 2), nullable=False),
    StructField("interest_rate", DoubleType(),       nullable=False),
    StructField("tenure_months", IntegerType(),      nullable=False),
    StructField("disbursed_on",  DateType(),         nullable=False),
    StructField("status",        StringType(),       nullable=False),
])

# multiLine=true — each JSON document spans multiple lines
loan_accounts = (
    spark.read
    .option("multiLine", "true")
    .schema(loan_accounts_schema)
    .json(f"{DATA}/loan_accounts")
)

print(f"loan_accounts: {loan_accounts.count()} rows")
loan_accounts.show(5)

## Reading Parquet

Parquet embeds the schema in the file footer — Spark reads it automatically. You can still pass an explicit schema to enforce types or select a subset of columns (predicate pushdown applies).

In [ ]:
card_transactions_schema = StructType([
    StructField("txn_id",       StringType(),       nullable=False),
    StructField("card_id",      StringType(),       nullable=False),
    StructField("customer_id",  StringType(),       nullable=False),
    StructField("amount",       DecimalType(18, 2), nullable=False),
    StructField("merchant",     StringType(),       nullable=True),
    StructField("category",     StringType(),       nullable=True),
    StructField("txn_ts",       TimestampType(),    nullable=False),
    StructField("status",       StringType(),       nullable=False),
    StructField("is_fraud",     BooleanType(),      nullable=False),
])

card_transactions = (
    spark.read
    .schema(card_transactions_schema)
    .parquet(f"{DATA}/card_transactions")
)

print(f"card_transactions: {card_transactions.count()} rows")
card_transactions.show(5)

## Reading ORC

ORC (Optimized Row Columnar) is the default output format for many legacy Hadoop pipelines. Like Parquet, the schema is embedded — pass an explicit schema to be safe.

In [ ]:
bank_accounts_schema = StructType([
    StructField("account_id",   StringType(),       nullable=False),
    StructField("customer_id",  StringType(),       nullable=False),
    StructField("account_type", StringType(),       nullable=False),
    StructField("balance",      DecimalType(18, 2), nullable=False),
    StructField("branch_code",  StringType(),       nullable=True),
    StructField("ifsc",         StringType(),       nullable=True),
    StructField("opened_on",    DateType(),         nullable=False),
    StructField("is_active",    BooleanType(),      nullable=False),
])

account_transactions_schema = StructType([
    StructField("txn_id",        StringType(),       nullable=False),
    StructField("account_id",    StringType(),       nullable=False),
    StructField("customer_id",   StringType(),       nullable=False),
    StructField("txn_type",      StringType(),       nullable=False),
    StructField("amount",        DecimalType(18, 2), nullable=False),
    StructField("balance_after", DecimalType(18, 2), nullable=False),
    StructField("description",   StringType(),       nullable=True),
    StructField("txn_ts",        TimestampType(),    nullable=False),
])

bank_accounts = (
    spark.read
    .schema(bank_accounts_schema)
    .orc(f"{DATA}/bank_accounts")
)

account_transactions = (
    spark.read
    .schema(account_transactions_schema)
    .orc(f"{DATA}/account_transactions")
)

print(f"bank_accounts       : {bank_accounts.count()} rows")
print(f"account_transactions: {account_transactions.count()} rows")
bank_accounts.show(3)

## Reading Delta Lake

Delta tables are directories — pass the path to the folder (not to individual files). Delta embeds schema, stats, and transaction log; no options are usually needed.

In [ ]:
payments_schema = StructType([
    StructField("payment_id",   StringType(),       nullable=False),
    StructField("customer_id",  StringType(),       nullable=False),
    StructField("payment_mode", StringType(),       nullable=False),
    StructField("amount",       DecimalType(18, 2), nullable=False),
    StructField("sender_vpa",   StringType(),       nullable=True),
    StructField("receiver_vpa", StringType(),       nullable=True),
    StructField("status",       StringType(),       nullable=False),
    StructField("is_fraud",     BooleanType(),      nullable=False),
    StructField("initiated_at", TimestampType(),    nullable=False),
    StructField("settled_at",   TimestampType(),    nullable=True),
])

payments = (
    spark.read
    .format("delta")
    .load(f"{DATA}/payments")
)

print(f"payments: {payments.count()} rows")
payments.show(5)

## Schema Inference vs Explicit Schema

Inference requires Spark to scan the data to guess types. It is convenient for exploration but risky in production — a single malformed row can change an inferred type silently.

| | Explicit schema | Inferred schema |
|---|---|---|
| Speed | Fast — no scan | Slow — full or sampled scan |
| Type safety | Guaranteed | Best-effort guess |
| Null handling | Declared per column | Assumed nullable |
| Recommended for | Production pipelines | Exploratory work only |

In [ ]:
# Inference — Spark scans the CSV to guess types
customers_inferred = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA}/customers")
)

print("Inferred schema:")
customers_inferred.printSchema()

print("\nExplicit schema:")
customers.printSchema()

# Compare: date_of_birth and created_at are often inferred as StringType

## Write Modes

Every `df.write` call requires a mode that controls what happens when the destination already exists:

| Mode | Behaviour |
|---|---|
| `overwrite` | Replace existing data completely |
| `append` | Add rows to existing data |
| `ignore` | Do nothing if destination exists (no error) |
| `error` (default) | Raise an error if destination exists |

## Writing Data

In [ ]:
from pyspark.sql.functions import col

# Write fraud-flagged card transactions to CSV for the compliance team
fraud_txns = card_transactions.filter(col("is_fraud") == True)

(
    fraud_txns
    .coalesce(1)                          # single output file — small result set
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(f"{OUT}/fraud_transactions_csv")
)
print("Fraud transactions written to CSV.")
print(f"Fraud count: {fraud_txns.count()}")

In [ ]:
# Write active loan accounts to JSON
active_loans = loan_accounts.filter(col("status") == "ACTIVE")

(
    active_loans
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(f"{OUT}/active_loans_json")
)
print(f"Active loans written to JSON: {active_loans.count()} rows")

In [ ]:
# Write card transactions to Parquet — efficient for downstream analytics
(
    card_transactions
    .write
    .mode("overwrite")
    .parquet(f"{OUT}/card_transactions_parquet")
)
print("Card transactions written to Parquet.")

## Partitioned Writes

`partitionBy` splits the output into subdirectories by column value — queries that filter on that column skip irrelevant partitions entirely (partition pruning).

```
data/card_transactions_by_status/
    status=SUCCESS/   part-0.parquet
    status=DECLINED/  part-0.parquet
    status=REVERSED/  part-0.parquet
```

In [ ]:
# Partition card transactions by status — downstream queries that filter on status
# will only read the relevant partition directory
(
    card_transactions
    .write
    .mode("overwrite")
    .partitionBy("status")
    .parquet(f"{OUT}/card_transactions_by_status")
)

# Verify: read back a single partition
success_txns = (
    spark.read
    .parquet(f"{OUT}/card_transactions_by_status/status=SUCCESS")
)
print(f"SUCCESS transactions: {success_txns.count()}")

# Read with partition column included
all_back = spark.read.parquet(f"{OUT}/card_transactions_by_status")
all_back.groupBy("status").count().orderBy("status").show()

## `coalesce` vs `repartition` Before Writing

Both control the number of output files. The choice matters for write performance and downstream read parallelism.

| | `coalesce(n)` | `repartition(n)` |
|---|---|---|
| Shuffle | No (narrow transformation) | Yes (full shuffle) |
| When to use | Reducing partition count | Increasing or balancing partitions |
| Output files | n files, possibly unequal size | n files, roughly equal size |

In [ ]:
# coalesce(1) — produce a single output file (no shuffle)
# Use for small exports that need one file (e.g. CSV report for Excel)
(
    customers
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(f"{OUT}/customers_single_file")
)
print(f"customers written as 1 file")

# repartition(4) — balance data evenly across 4 files (full shuffle)
# Use before writing large Parquet tables to ensure equal partition sizes
(
    card_transactions
    .repartition(4)
    .write.mode("overwrite")
    .parquet(f"{OUT}/card_transactions_4_parts")
)
print(f"card_transactions written as 4 balanced partitions")

## Summary

- `spark.read.format(x).option(…).load(path)` and shorthand `.csv()`, `.json()`, `.parquet()`, `.orc()` all use the same unified API.
- Always pass an explicit schema in production — inference is slower and can silently change types.
- CSV needs `option("header", "true")`; JSON multiline documents need `option("multiLine", "true")`; Parquet and ORC embed their schema; Delta needs only `format("delta")`.
- Write modes: `overwrite` replaces, `append` adds, `ignore` skips, `error` (default) raises on conflict.
- `partitionBy` creates subdirectories per column value — downstream filters on that column skip irrelevant partitions.
- `coalesce(n)` reduces files without a shuffle; `repartition(n)` balances files with a full shuffle.